# 03B — Per-region analysis  ·  line B (DS0N)

The `sebastian_sun_spots` datasets, one at a time, through **the same eight steps as
`03A_data_analysis.ipynb`**. Read the two side by side: the step numbers, the function calls
and the figures are the same, because both loaders return the same `data` dict.

What differs is only the entry point and the thresholds:

|  | line A (NOAA) | line B (DS0N) |
|---|---|---|
| loader | `loaders.load_noaa_region` | `loaders.load_ds0n_region` |
| on disk | `region_01_*_cube.fits`, on a uniform grid written by `02A` | IDL-style `cube_*.fits` + `.sav` timing, as delivered |
| thresholds | fractions of each frame's own quiet sun | absolute DN (30 000 / 50 000) |
| corrections | applied here, by `02A` | already applied before delivery |

The two keep **separate loaders on purpose**. Their thresholds mean different things, and
trying to serve both from one function is how the old `sunspot_analysis.py` ended up carrying
two whole pipeline generations at once.

This notebook replaces `03T_data_analisis.ipynb` (now in `archive/`), which was twelve
copy-pasted blocks with the thresholds retyped in each. Here every block reads
`config.params_for(ds_dir, line='B')`.

### Which datasets

`DS00`–`DS09` load. **`DS10`'s `cube_continuum.fits` is truncated** — astropy refuses it with
*"buffer is too small for requested array"* — and **`DS11` has no continuum cube at all**.
Both are listed in `config.DS0N_BROKEN`; re-deliver those two files and add them back to
`config.DS0N_IDS`.

In [ ]:
import pathlib
import sys

project_root = pathlib.Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root.resolve()))

import numpy as np
from IPython.display import FileLink

from src import analysis, animation, config, oscillation, spectra
from src.loaders import load_ds0n_region

datasets = config.ds0n_regions()
print(f'{len(datasets)} DS0N dataset(s) with a continuum cube:')
for i, ds_dir in enumerate(datasets):
    print(f'  [{i}] {ds_dir.name}')
for name, why in config.DS0N_BROKEN.items():
    print(f'  --  {name}: {why}')

print('\nThresholds (src/config.py, DS0N_DEFAULTS):')
for key, value in config.DS0N_DEFAULTS.items():
    print(f'  {key:20s} = {value!r}')

## Knobs for this run

The same three as `03A`, with the same meanings.

In [ ]:
SAVE = False
DOMINANT_PERIOD_MIN = 1440.0
RUN_MU = False

---

# DS00

### Step 1 — Verify the cadence, load the cubes, build the regions. `params` comes from `config.DS0N_DEFAULTS`.

In [ ]:
DS_INDEX = 0

ds_dir     = datasets[DS_INDEX]
out_dir    = config.DS0N_PROCESSED_DIR / ds_dir.name
plots_dir  = out_dir / 'plots'

params = config.params_for(ds_dir, line='B')
view   = config.view_params(params)

from src.loaders import verify_cadence
verify_cadence(ds_dir, expected_s=params['cadence_s'])

data = load_ds0n_region(ds_dir, **config.loader_kwargs(params))

print(ds_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual, written to `metrics.csv` for `04`.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, out_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

### Step 4 — Raw, normalised, quiet-sun-subtracted and magnetogram-residual views.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra.

In [ ]:
cadence_s = data['cadence_s']

fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# DS0N ships `cube_mu.fits`, so the geometry check is cheap here — no map loading needed,
# unlike line A where mu has to be computed frame by frame.
if RUN_MU and (ds_dir / 'cube_mu.fits').exists():
    cube_mu = analysis.mu_cube_ds0n(ds_dir)
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
elif not RUN_MU:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')
else:
    print(f'{ds_dir.name} has no cube_mu.fits')

### Step 8 — Animation.

In [ ]:
anim_path = animation.save_animation(data, metrics, out_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# DS01

### Step 1 — Verify the cadence, load the cubes, build the regions. `params` comes from `config.DS0N_DEFAULTS`.

In [ ]:
DS_INDEX = 1

ds_dir     = datasets[DS_INDEX]
out_dir    = config.DS0N_PROCESSED_DIR / ds_dir.name
plots_dir  = out_dir / 'plots'

params = config.params_for(ds_dir, line='B')
view   = config.view_params(params)

from src.loaders import verify_cadence
verify_cadence(ds_dir, expected_s=params['cadence_s'])

data = load_ds0n_region(ds_dir, **config.loader_kwargs(params))

print(ds_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual, written to `metrics.csv` for `04`.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, out_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

### Step 4 — Raw, normalised, quiet-sun-subtracted and magnetogram-residual views.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra.

In [ ]:
cadence_s = data['cadence_s']

fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# DS0N ships `cube_mu.fits`, so the geometry check is cheap here — no map loading needed,
# unlike line A where mu has to be computed frame by frame.
if RUN_MU and (ds_dir / 'cube_mu.fits').exists():
    cube_mu = analysis.mu_cube_ds0n(ds_dir)
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
elif not RUN_MU:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')
else:
    print(f'{ds_dir.name} has no cube_mu.fits')

### Step 8 — Animation.

In [ ]:
anim_path = animation.save_animation(data, metrics, out_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# DS02

### Step 1 — Verify the cadence, load the cubes, build the regions. `params` comes from `config.DS0N_DEFAULTS`.

In [ ]:
DS_INDEX = 2

ds_dir     = datasets[DS_INDEX]
out_dir    = config.DS0N_PROCESSED_DIR / ds_dir.name
plots_dir  = out_dir / 'plots'

params = config.params_for(ds_dir, line='B')
view   = config.view_params(params)

from src.loaders import verify_cadence
verify_cadence(ds_dir, expected_s=params['cadence_s'])

data = load_ds0n_region(ds_dir, **config.loader_kwargs(params))

print(ds_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual, written to `metrics.csv` for `04`.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, out_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

### Step 4 — Raw, normalised, quiet-sun-subtracted and magnetogram-residual views.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra.

In [ ]:
cadence_s = data['cadence_s']

fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# DS0N ships `cube_mu.fits`, so the geometry check is cheap here — no map loading needed,
# unlike line A where mu has to be computed frame by frame.
if RUN_MU and (ds_dir / 'cube_mu.fits').exists():
    cube_mu = analysis.mu_cube_ds0n(ds_dir)
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
elif not RUN_MU:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')
else:
    print(f'{ds_dir.name} has no cube_mu.fits')

### Step 8 — Animation.

In [ ]:
anim_path = animation.save_animation(data, metrics, out_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# DS03

### Step 1 — Verify the cadence, load the cubes, build the regions. `params` comes from `config.DS0N_DEFAULTS`.

In [ ]:
DS_INDEX = 3

ds_dir     = datasets[DS_INDEX]
out_dir    = config.DS0N_PROCESSED_DIR / ds_dir.name
plots_dir  = out_dir / 'plots'

params = config.params_for(ds_dir, line='B')
view   = config.view_params(params)

from src.loaders import verify_cadence
verify_cadence(ds_dir, expected_s=params['cadence_s'])

data = load_ds0n_region(ds_dir, **config.loader_kwargs(params))

print(ds_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual, written to `metrics.csv` for `04`.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, out_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

### Step 4 — Raw, normalised, quiet-sun-subtracted and magnetogram-residual views.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra.

In [ ]:
cadence_s = data['cadence_s']

fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# DS0N ships `cube_mu.fits`, so the geometry check is cheap here — no map loading needed,
# unlike line A where mu has to be computed frame by frame.
if RUN_MU and (ds_dir / 'cube_mu.fits').exists():
    cube_mu = analysis.mu_cube_ds0n(ds_dir)
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
elif not RUN_MU:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')
else:
    print(f'{ds_dir.name} has no cube_mu.fits')

### Step 8 — Animation.

In [ ]:
anim_path = animation.save_animation(data, metrics, out_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# DS04

### Step 1 — Verify the cadence, load the cubes, build the regions. `params` comes from `config.DS0N_DEFAULTS`.

In [ ]:
DS_INDEX = 4

ds_dir     = datasets[DS_INDEX]
out_dir    = config.DS0N_PROCESSED_DIR / ds_dir.name
plots_dir  = out_dir / 'plots'

params = config.params_for(ds_dir, line='B')
view   = config.view_params(params)

from src.loaders import verify_cadence
verify_cadence(ds_dir, expected_s=params['cadence_s'])

data = load_ds0n_region(ds_dir, **config.loader_kwargs(params))

print(ds_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual, written to `metrics.csv` for `04`.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, out_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

### Step 4 — Raw, normalised, quiet-sun-subtracted and magnetogram-residual views.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra.

In [ ]:
cadence_s = data['cadence_s']

fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# DS0N ships `cube_mu.fits`, so the geometry check is cheap here — no map loading needed,
# unlike line A where mu has to be computed frame by frame.
if RUN_MU and (ds_dir / 'cube_mu.fits').exists():
    cube_mu = analysis.mu_cube_ds0n(ds_dir)
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
elif not RUN_MU:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')
else:
    print(f'{ds_dir.name} has no cube_mu.fits')

### Step 8 — Animation.

In [ ]:
anim_path = animation.save_animation(data, metrics, out_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# DS05

### Step 1 — Verify the cadence, load the cubes, build the regions. `params` comes from `config.DS0N_DEFAULTS`.

In [ ]:
DS_INDEX = 5

ds_dir     = datasets[DS_INDEX]
out_dir    = config.DS0N_PROCESSED_DIR / ds_dir.name
plots_dir  = out_dir / 'plots'

params = config.params_for(ds_dir, line='B')
view   = config.view_params(params)

from src.loaders import verify_cadence
verify_cadence(ds_dir, expected_s=params['cadence_s'])

data = load_ds0n_region(ds_dir, **config.loader_kwargs(params))

print(ds_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual, written to `metrics.csv` for `04`.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, out_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

### Step 4 — Raw, normalised, quiet-sun-subtracted and magnetogram-residual views.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra.

In [ ]:
cadence_s = data['cadence_s']

fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# DS0N ships `cube_mu.fits`, so the geometry check is cheap here — no map loading needed,
# unlike line A where mu has to be computed frame by frame.
if RUN_MU and (ds_dir / 'cube_mu.fits').exists():
    cube_mu = analysis.mu_cube_ds0n(ds_dir)
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
elif not RUN_MU:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')
else:
    print(f'{ds_dir.name} has no cube_mu.fits')

### Step 8 — Animation.

In [ ]:
anim_path = animation.save_animation(data, metrics, out_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# DS06

### Step 1 — Verify the cadence, load the cubes, build the regions. `params` comes from `config.DS0N_DEFAULTS`.

In [ ]:
DS_INDEX = 6

ds_dir     = datasets[DS_INDEX]
out_dir    = config.DS0N_PROCESSED_DIR / ds_dir.name
plots_dir  = out_dir / 'plots'

params = config.params_for(ds_dir, line='B')
view   = config.view_params(params)

from src.loaders import verify_cadence
verify_cadence(ds_dir, expected_s=params['cadence_s'])

data = load_ds0n_region(ds_dir, **config.loader_kwargs(params))

print(ds_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual, written to `metrics.csv` for `04`.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, out_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

### Step 4 — Raw, normalised, quiet-sun-subtracted and magnetogram-residual views.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra.

In [ ]:
cadence_s = data['cadence_s']

fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# DS0N ships `cube_mu.fits`, so the geometry check is cheap here — no map loading needed,
# unlike line A where mu has to be computed frame by frame.
if RUN_MU and (ds_dir / 'cube_mu.fits').exists():
    cube_mu = analysis.mu_cube_ds0n(ds_dir)
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
elif not RUN_MU:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')
else:
    print(f'{ds_dir.name} has no cube_mu.fits')

### Step 8 — Animation.

In [ ]:
anim_path = animation.save_animation(data, metrics, out_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# DS07

### Step 1 — Verify the cadence, load the cubes, build the regions. `params` comes from `config.DS0N_DEFAULTS`.

In [ ]:
DS_INDEX = 7

ds_dir     = datasets[DS_INDEX]
out_dir    = config.DS0N_PROCESSED_DIR / ds_dir.name
plots_dir  = out_dir / 'plots'

params = config.params_for(ds_dir, line='B')
view   = config.view_params(params)

from src.loaders import verify_cadence
verify_cadence(ds_dir, expected_s=params['cadence_s'])

data = load_ds0n_region(ds_dir, **config.loader_kwargs(params))

print(ds_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual, written to `metrics.csv` for `04`.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, out_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

### Step 4 — Raw, normalised, quiet-sun-subtracted and magnetogram-residual views.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra.

In [ ]:
cadence_s = data['cadence_s']

fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# DS0N ships `cube_mu.fits`, so the geometry check is cheap here — no map loading needed,
# unlike line A where mu has to be computed frame by frame.
if RUN_MU and (ds_dir / 'cube_mu.fits').exists():
    cube_mu = analysis.mu_cube_ds0n(ds_dir)
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
elif not RUN_MU:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')
else:
    print(f'{ds_dir.name} has no cube_mu.fits')

### Step 8 — Animation.

In [ ]:
anim_path = animation.save_animation(data, metrics, out_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# DS08

### Step 1 — Verify the cadence, load the cubes, build the regions. `params` comes from `config.DS0N_DEFAULTS`.

In [ ]:
DS_INDEX = 8

ds_dir     = datasets[DS_INDEX]
out_dir    = config.DS0N_PROCESSED_DIR / ds_dir.name
plots_dir  = out_dir / 'plots'

params = config.params_for(ds_dir, line='B')
view   = config.view_params(params)

from src.loaders import verify_cadence
verify_cadence(ds_dir, expected_s=params['cadence_s'])

data = load_ds0n_region(ds_dir, **config.loader_kwargs(params))

print(ds_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual, written to `metrics.csv` for `04`.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, out_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

### Step 4 — Raw, normalised, quiet-sun-subtracted and magnetogram-residual views.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra.

In [ ]:
cadence_s = data['cadence_s']

fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# DS0N ships `cube_mu.fits`, so the geometry check is cheap here — no map loading needed,
# unlike line A where mu has to be computed frame by frame.
if RUN_MU and (ds_dir / 'cube_mu.fits').exists():
    cube_mu = analysis.mu_cube_ds0n(ds_dir)
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
elif not RUN_MU:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')
else:
    print(f'{ds_dir.name} has no cube_mu.fits')

### Step 8 — Animation.

In [ ]:
anim_path = animation.save_animation(data, metrics, out_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# DS09

### Step 1 — Verify the cadence, load the cubes, build the regions. `params` comes from `config.DS0N_DEFAULTS`.

In [ ]:
DS_INDEX = 9

ds_dir     = datasets[DS_INDEX]
out_dir    = config.DS0N_PROCESSED_DIR / ds_dir.name
plots_dir  = out_dir / 'plots'

params = config.params_for(ds_dir, line='B')
view   = config.view_params(params)

from src.loaders import verify_cadence
verify_cadence(ds_dir, expected_s=params['cadence_s'])

data = load_ds0n_region(ds_dir, **config.loader_kwargs(params))

print(ds_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual, written to `metrics.csv` for `04`.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, out_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

### Step 4 — Raw, normalised, quiet-sun-subtracted and magnetogram-residual views.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra.

In [ ]:
cadence_s = data['cadence_s']

fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# DS0N ships `cube_mu.fits`, so the geometry check is cheap here — no map loading needed,
# unlike line A where mu has to be computed frame by frame.
if RUN_MU and (ds_dir / 'cube_mu.fits').exists():
    cube_mu = analysis.mu_cube_ds0n(ds_dir)
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
elif not RUN_MU:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')
else:
    print(f'{ds_dir.name} has no cube_mu.fits')

### Step 8 — Animation.

In [ ]:
anim_path = animation.save_animation(data, metrics, out_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

## Next

`04_data_comparison.ipynb` reads the `metrics.csv` files written in step 3 and compares the
datasets to each other — and to line A.